# Hybrid Retrieval Test
**Three signals → RRF fusion**
- Signal 1: KG traversal (NetworkX graph)
- Signal 2: Dense vector search (Qdrant)
- Signal 3: BM25 keyword search

Run cells top to bottom. Change `QUERY` in the last section to test different questions.

In [ ]:
import sys, os
# Notebook lives at src/notebooks/ — one level up is src/
sys.path.insert(0, os.path.abspath('..'))

import re
import numpy as np
from pathlib import Path
from rank_bm25 import BM25Okapi

from config.config import KGConfig, AdvancedConfig
from data.chunk_loader import load_preprocessed_chunks
from indexing.kg_store import KGStore
from indexing.kg_retriever import KGRetriever
from indexing.embedder import TextEmbedder
from indexing.vector_database import QdrantVectorDB

KG_CFG  = KGConfig()
VEC_CFG = AdvancedConfig()
print('Config loaded')

## 1 — Load the Knowledge Graph

In [ ]:
graph, kg_chunks = KGStore.load(KG_CFG.KG_GRAPH_FILE)
kg_retriever = KGRetriever(graph=graph, chunks=kg_chunks, hops=2)

print(f'Nodes : {graph.number_of_nodes():,}')
print(f'Edges : {graph.number_of_edges():,}')
print(f'Chunks: {len(kg_chunks):,}')

## 2 — Build BM25 Index

In [ ]:
def tokenize(text):
    return re.findall(r'\w+', text.lower())

chunks = load_preprocessed_chunks(Path(KG_CFG.PREPROCESSED_CHUNKS_FILE))
bm25 = BM25Okapi([tokenize(c['text']) for c in chunks])
print(f'BM25 index built over {len(chunks):,} chunks')

## 3 — Connect to Qdrant + Embedder

In [ ]:

# The config default points to a collection that doesn't exist locally.
# Override to the populated one (19,888 text points).
VEC_CFG.VECTOR_DB_COLLECTION = "advanced_fixed_size"

embedder  = TextEmbedder(VEC_CFG.EMBEDDING_MODEL)
vector_db = QdrantVectorDB(VEC_CFG)
print(f'Qdrant collection : {VEC_CFG.VECTOR_DB_COLLECTION}')
print(f'Documents in index: {vector_db.count_documents():,}')


## 4 — RRF Fusion Helper

In [ ]:
RRF_K = 60

def rrf_fuse(ranked_lists, top_k=5):
    """
    ranked_lists: list of lists, each list is [{id, text, metadata, ...}, ...]
                  ordered best-first.
    Returns the top_k results fused by Reciprocal Rank Fusion.
    """
    scores = {}
    docs   = {}

    for ranked in ranked_lists:
        for rank, doc in enumerate(ranked):
            uid = doc.get('uid') or doc.get('id') or doc['text'][:60]
            scores[uid] = scores.get(uid, 0) + 1 / (RRF_K + rank)
            docs[uid]   = doc

    top_uids = sorted(scores, key=scores.get, reverse=True)[:top_k]
    return [{'rrf_score': scores[u], **docs[u]} for u in top_uids]

## 5 — Run a Query
Change `QUERY` and `TOP_K` as needed.

In [ ]:
QUERY = "What is the relationship between ISIS and terrorism?"
TOP_K = 5
N_CANDIDATES = TOP_K * 10  # fetch more per signal before fusing

In [ ]:
# ── Signal 1: KG traversal ───────────────────────────────────────────────────
kg_results = kg_retriever.retrieve(QUERY, top_k=N_CANDIDATES)
# normalise shape: add uid key for RRF
for r in kg_results:
    r['uid'] = r['metadata']['uid']

# ── Signal 2: Dense (Qdrant) ─────────────────────────────────────────────────
query_emb    = embedder.embed_query(QUERY)
dense_results = vector_db.retrieve(query_emb, top_k=N_CANDIDATES, allowed_types=['text'])
for r in dense_results:
    r['uid'] = str(r['id'])

# ── Signal 3: BM25 ───────────────────────────────────────────────────────────
bm25_scores  = bm25.get_scores(tokenize(QUERY))
bm25_top_ids = np.argsort(bm25_scores)[::-1][:N_CANDIDATES].tolist()
bm25_results = []
for idx in bm25_top_ids:
    c = chunks[idx]
    bm25_results.append({
        'uid'   : f"{c['pdf_name']}::{c['chunk_id']}",
        'text'  : c['text'],
        'score' : float(bm25_scores[idx]),
        'metadata': {'pdf_name': c['pdf_name'], 'chunk_id': c['chunk_id']},
    })

print(f'KG results   : {len(kg_results)}')
print(f'Dense results: {len(dense_results)}')
print(f'BM25 results : {len(bm25_results)}')

## 6 — Inspect Each Signal Individually

In [ ]:
def show(label, results, n=3):
    print(f'\n── {label} ──')
    for i, r in enumerate(results[:n]):
        meta = r.get('metadata') or r.get('payload') or {}
        pdf  = meta.get('pdf_name', '?')
        cid  = meta.get('chunk_id', '?')
        text = r.get('text','')[:120].replace('\n',' ')
        print(f'  [{i+1}] {pdf} | chunk {cid}')
        print(f'       {text}...')

show('KG traversal', kg_results)
show('Dense (Qdrant)', dense_results)
show('BM25', bm25_results)

## 7 — Fuse with RRF

In [ ]:
fused = rrf_fuse([kg_results, dense_results, bm25_results], top_k=TOP_K)

print(f'\n── Fused top-{TOP_K} (RRF) ──')
for i, r in enumerate(fused):
    meta = r.get('metadata') or r.get('payload') or {}
    pdf  = meta.get('pdf_name', '?')
    cid  = meta.get('chunk_id', '?')
    text = r.get('text','')[:150].replace('\n',' ')
    print(f'  [{i+1}] rrf={r["rrf_score"]:.4f} | {pdf} | chunk {cid}')
    print(f'       {text}...')
    print()